# LCDM MCMC: paper-style BAO, BBN, Planck anisotropies, and CMB lensing

This notebook rewrites the earlier verification driver so the sampled cosmology basis and prior logic follow the ACT DR6 lensing paper more closely.

It can run the following combinations:
- `BAO + BBN`
- `Planck anisotropies`
- `ACT DR6 lensing + BBN`
- `Planck lensing + BAO + BBN`
- `ACT DR6 lensing + BAO + BBN`

The sampled cosmology basis is `100theta_MC`, `ombh2`, `omch2`, `logA`, `ns`, plus `tau` only for the Planck-primary run. CMB nuisance parameters are sampled per likelihood combination and use the internal `candl` or `clipy` priors.

Two caveats matter for interpretation:
- `100theta_MC` is implemented here with the repo's differentiable `theta_star` background approximation, so it is a smooth surrogate for the paper basis rather than CAMB's exact `theta_MC`.
- `BAO + BBN` does not constrain the amplitude direction by itself, so its `sigma8` projection is prior-dominated. Keep that curve as a sanity check, not as a precision validation target.


In [1]:
import os
from pathlib import Path

os.environ.setdefault('XDG_CACHE_HOME', str(Path.cwd() / '.cache'))
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.cache' / 'matplotlib'))
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

import jax
jax.config.update('jax_enable_x64', True)

import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from numpy.linalg import inv
from scipy.ndimage import gaussian_filter

import candl_data
from ps_1loop_jax import background as bg

from jaxptpolypol.bao import load_desi_dr2, make_bao_theory_fn
from jaxptpolypol.cmb import (
    CandlParameterLayout,
    get_candl_default_parameters,
    get_candl_parameter_names,
    load_candl_likelihood,
    make_candl_loglike_fn,
    make_candl_pars_to_theory_specs_fn,
)
from jaxptpolypol.derived import sigma8_from_linear_pk
from jaxptpolypol.model import CosmoEmulator
from jaxptpolypol.params import CosmoParams
from jaxptpolypol.sampler import make_transform, run_nuts, samples_to_physical


In [2]:
REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
for _base in REPO_ROOT_CANDIDATES:
    if (_base / 'ext_data/bao_data/desi_bao_dr2').exists():
        REPO_ROOT = _base
        break
else:
    REPO_ROOT = Path.cwd()

PLANCK_ROOT = Path('/Users/nguyenmn/candl/clipy/Planck_likelihoods/baseline/plc_3.0')
PLANCK_HIGHL = PLANCK_ROOT / 'hi_l/plik/plik_rd12_HM_v22b_TTTEEE.clik'
PLANCK_LOWL_EE = PLANCK_ROOT / 'low_l/simall/simall_100x143_offlike5_EE_Aplanck_B.clik'
PLANCK_LENSING = PLANCK_ROOT / 'lensing/smicadx12_Dec5_ftl_mv2_ndclpp_p_teb_consext8_CMBmarged.clik_lensing'
ACT_DR6_LENS = candl_data.ACT_DR6_Lens  # Index shortcut; upstream default resolves to lens_only. Use variant='use_CMB' or candl_data.ACT_DR6_Lens_and_CMB for the combined ACT lensing+CMB case.

EMULATOR_DIR = Path('/Users/nguyenmn/cosmopower-jax-for-pfs/cosmology/jense2024/jense_2023_camb_lcdm/networks')
CMB_EMULATOR_FILENAMES = {
    'TT': str(EMULATOR_DIR / 'jense_2023_camb_lcdm_Cl_tt.npz'),
    'TE': str(EMULATOR_DIR / 'jense_2023_camb_lcdm_Cl_te.npz'),
    'EE': str(EMULATOR_DIR / 'jense_2023_camb_lcdm_Cl_ee.npz'),
    'pp': str(EMULATOR_DIR / 'jense_2023_camb_lcdm_Cl_pp.npz'),
}
PKLIN_EMULATOR_PATH = str(EMULATOR_DIR / 'jense_2023_camb_lcdm_Pk_lin.npz')
BAO_DATA_DIR = REPO_ROOT / 'ext_data/bao_data/desi_bao_dr2'

COSMO_KEYS_NATIVE = ('H0', 'ombh2', 'omch2', 'logA', 'ns', 'tau')
COSMO_SIZES_NATIVE = (1, 1, 1, 1, 1, 1)
NONPLANCK_SAMPLED_COSMO_KEYS = ('100theta', 'ombh2', 'omch2', 'logA', 'ns')
PLANCK_SAMPLED_COSMO_KEYS = NONPLANCK_SAMPLED_COSMO_KEYS + ('tau',)
FIDUCIAL_NATIVE = {
    'H0': 67.32,
    'ombh2': 0.02237,
    'omch2': 0.1200,
    'logA': 3.044,
    'ns': 0.9649,
    'tau': 0.0544,
}
MNU_FIXED = 0.06
NEFF_FIXED = 3.046

MARGINALIZE_CMB_NUISANCE = False
# Use fixed CMB nuisance parameters for the sigma8-Omega_m verification sweep.
# Turning this on is substantially slower and was not needed to land near the paper contours.
INCLUDE_INTERNAL_CMB_PRIORS = True
INCLUDE_PAPER_REFERENCE_MARKERS = True

RUN_BAO_BBN = True
RUN_PLANCK_PRIMARY = True
RUN_ACT_LENS_BBN = True
RUN_PLANCK_LENSING_BAO = True
RUN_ACT_LENSING_BAO = True
RUN_ACT_PLANCK_LENSING_BAO = True

LAPTOP_MODE = True
if LAPTOP_MODE:
    NUM_WARMUP = 100
    NUM_SAMPLES = 200
    NUM_CHAINS = 2
    SCAN_CHUNK = 16
    MAX_TREE_DEPTH = (8, 8)
    DERIVED_CHUNK_SIZE = 64
else:
    NUM_WARMUP = 200
    NUM_SAMPLES = 800
    NUM_CHAINS = 4
    SCAN_CHUNK = 256
    MAX_TREE_DEPTH = (10, 10)
    DERIVED_CHUNK_SIZE = 512

MOSSABBN_MEAN = 0.02233
MOSSABBN_SIGMA = 0.00036
NS_PRIOR_MEAN = 0.96
NS_PRIOR_SIGMA = 0.02

COSMO_FALLBACK_SCALES = {
    '100theta': 0.005,
    'H0': 5.0,
    'ombh2': 3.6e-4,
    'omch2': 6.0e-3,
    'logA': 5.0e-2,
    'ns': 2.0e-2,
    'tau': 1.0e-2,
}

MANUAL_WHITENING_SCALES = {
    'ACT DR6 lensing + BBN': {
        '100theta': 0.02,
        'ombh2': 4.5e-4,
        'omch2': 2.0e-2,
        'logA': 0.12,
        'ns': 0.03,
    },
}

PAPER_REFERENCE_POINTS = {
    'Planck anisotropies': {
        'sigma8': 0.811,
        'Omega_m': 0.314,
        'H0': 67.3,
        'source': 'Table 2: Planck CMB aniso. (PR4 TT+TE+EE) + SRoll2 low-ell EE',
    },
    'ACT DR6 lensing + BAO + BBN': {
        'sigma8': 0.820,
        'S8': 0.840,
        'Omega_m': 0.315,
        'H0': 68.2,
        'source': 'Table 2: ACT CMB lensing + BAO',
    },
    'ACT + Planck lensing + BAO + BBN': {
        'sigma8': 0.815,
        'S8': 0.830,
        'Omega_m': 0.312,
        'H0': 68.1,
        'source': 'Table 2: ACT+Planck lensing + BAO',
    },
    'ACT + Planck lensing (extended) + BAO': {
        'sigma8': 0.820,
        'S8': 0.841,
        'Omega_m': 0.316,
        'H0': 68.3,
        'source': 'Table 2: ACT+Planck lensing (extended) + BAO',
        'color': 'black',
    },
}

COMBINATION_COLORS = {
    'BAO + BBN': 'tab:gray',
    'Planck anisotropies': 'tab:blue',
    'ACT DR6 lensing + BBN': 'tab:green',
    'Planck lensing + BAO + BBN': 'tab:purple',
    'ACT DR6 lensing + BAO + BBN': 'tab:orange',
    'ACT + Planck lensing + BAO + BBN': 'tab:red',
}

PARAM_LABELS = {
    '100theta': r'$100\,\theta_\mathrm{MC}$',
    'ombh2': r'$\omega_b$',
    'omch2': r'$\omega_c$',
    'logA': r'$\log(10^{10} A_s)$',
    'ns': r'$n_s$',
    'tau': r'$\tau$',
    'Omega_m': r'$\Omega_m$',
    'sigma8': r'$\sigma_8$',
    'S8': r'$S_8$',
    'H0': r'$H_0$',
}

print('Planck high-ell:', PLANCK_HIGHL)
print('Planck low-ell EE:', PLANCK_LOWL_EE)
print('Planck lensing:', PLANCK_LENSING)
print('ACT DR6 lensing YAML:', ACT_DR6_LENS)
print('BAO data dir:', BAO_DATA_DIR)
print('CMB nuisance parameters sampled:', MARGINALIZE_CMB_NUISANCE)
print('Internal candl/clipy priors active:', INCLUDE_INTERNAL_CMB_PRIORS)
print('Mode:', 'LAPTOP' if LAPTOP_MODE else 'SERVER')
print(f'  warmup={NUM_WARMUP}, samples={NUM_SAMPLES}, chains={NUM_CHAINS}')
print(f'  scan_chunk={SCAN_CHUNK}, max_tree_depth={MAX_TREE_DEPTH}')
print(f'  derived_chunk_size={DERIVED_CHUNK_SIZE}')



Planck high-ell: /Users/nguyenmn/candl/clipy/Planck_likelihoods/baseline/plc_3.0/hi_l/plik/plik_rd12_HM_v22b_TTTEEE.clik
Planck low-ell EE: /Users/nguyenmn/candl/clipy/Planck_likelihoods/baseline/plc_3.0/low_l/simall/simall_100x143_offlike5_EE_Aplanck_B.clik
Planck lensing: /Users/nguyenmn/candl/clipy/Planck_likelihoods/baseline/plc_3.0/lensing/smicadx12_Dec5_ftl_mv2_ndclpp_p_teb_consext8_CMBmarged.clik_lensing
ACT DR6 lensing YAML: /Users/nguyenmn/candl/candl_data/candl_data/ACT_DR6_Lens_v1/ACT_DR6_KK_index.yaml
BAO data dir: /Users/nguyenmn/jaxPTPolyPol/ext_data/bao_data/desi_bao_dr2
CMB nuisance parameters sampled: False
Internal candl/clipy priors active: True
Mode: LAPTOP
  warmup=100, samples=200, chains=2
  scan_chunk=16, max_tree_depth=(8, 8)
  derived_chunk_size=64


## BAO data, theory backends, and likelihood loading

In [3]:
def scalar_value(value):
    return float(np.asarray(value).reshape(()))


def ordered_union(name_lists):
    ordered = []
    seen = set()
    for names in name_lists:
        for name in names:
            if name not in seen:
                seen.add(name)
                ordered.append(name)
    return tuple(ordered)


def summarize_scalar(samples):
    q16, q50, q84 = np.percentile(np.asarray(samples), [16.0, 50.0, 84.0])
    return {
        'q16': float(q16),
        'median': float(q50),
        'q84': float(q84),
        'minus': float(q50 - q16),
        'plus': float(q84 - q50),
        'mean': float(np.mean(samples)),
        'std': float(np.std(samples)),
    }


def load_likelihood_terms(include_internal_priors=True):
    clipy_args = {'all_priors': True} if include_internal_priors else {}
    return {
        'planck_highl': load_candl_likelihood(
            str(PLANCK_HIGHL),
            wrapper='clipy',
            additional_args=clipy_args,
        ),
        'planck_lowl_ee': load_candl_likelihood(
            str(PLANCK_LOWL_EE),
            wrapper='clipy',
            additional_args=clipy_args,
        ),
        'planck_lensing': load_candl_likelihood(
            str(PLANCK_LENSING),
            wrapper='clipy',
            additional_args=clipy_args,
        ),
        'act_dr6_lensing': load_candl_likelihood(
            ACT_DR6_LENS,
            lensing=True,
            feedback=False,
            clear_internal_priors=not include_internal_priors,
        ),
    }


bao_data = load_desi_dr2('all', data_dir=BAO_DATA_DIR)
pars_to_theory_specs = make_candl_pars_to_theory_specs_fn(
    emulator_filenames=CMB_EMULATOR_FILENAMES,
)
pklin_emulator = CosmoEmulator(probe='custom_log', emulator_path=PKLIN_EMULATOR_PATH)
PKLIN_FIXED_INPUTS = {
    'z': 0.0,
    'A_b': 3.13,
    'eta_b': 0.603,
    'logT_AGN': 7.8,
}
likelihoods = load_likelihood_terms(include_internal_priors=INCLUDE_INTERNAL_CMB_PRIORS)
MASTER_CMB_TERM_ORDER = ('planck_highl', 'planck_lowl_ee', 'planck_lensing', 'act_dr6_lensing')
TERM_INDEX = {name: i for i, name in enumerate(MASTER_CMB_TERM_ORDER)}

nuisance_names_by_term = {}
nuisance_defaults = {}
for term_name, like in likelihoods.items():
    names = get_candl_parameter_names(
        like,
        cosmo_keys=COSMO_KEYS_NATIVE,
        include_prior_params=True,
    )
    nuisance_names_by_term[term_name] = names
    for key, value in get_candl_default_parameters(like).items():
        nuisance_defaults.setdefault(key, scalar_value(value))

MASTER_CMB_NUISANCE_NAMES = (
    ordered_union(nuisance_names_by_term.get(term_name, ()) for term_name in MASTER_CMB_TERM_ORDER)
    if MARGINALIZE_CMB_NUISANCE else ()
)
MASTER_LAYOUT = CandlParameterLayout(
    cosmo_keys=COSMO_KEYS_NATIVE,
    cosmo_sizes=COSMO_SIZES_NATIVE,
    cmb_nuisance_names=tuple(MASTER_CMB_NUISANCE_NAMES),
)
MASTER_FIXED_CMB_PARAMS = {
    name: nuisance_defaults[name]
    for name in ordered_union(nuisance_names_by_term.get(term_name, ()) for term_name in MASTER_CMB_TERM_ORDER)
    if name not in set(MASTER_CMB_NUISANCE_NAMES)
}
CMB_TERM_LOGLIKES = {
    term_name: make_candl_loglike_fn(
        likelihoods[term_name],
        pars_to_theory_specs=pars_to_theory_specs,
        layout=MASTER_LAYOUT,
        fixed_cmb_params=MASTER_FIXED_CMB_PARAMS,
        jit_compile=True,
    )
    for term_name in MASTER_CMB_TERM_ORDER
}

print(f'Loaded {len(bao_data.data_points)} BAO points with covariance {bao_data.cov.shape}.')
print('Pk_lin emulator parameters:', tuple(pklin_emulator.parameters))
print('Fixed extra Pk emulator inputs:', PKLIN_FIXED_INPUTS)
print('Master CMB term order:', MASTER_CMB_TERM_ORDER)
print('Master nuisance block size:', len(MASTER_CMB_NUISANCE_NAMES))
print('Per-term CMB nuisance/prior scalars:')
for term_name, names in nuisance_names_by_term.items():
    print(f'  {term_name:>16s}: {len(names):3d}')



Tried to load pickle file from pre-trained model, but failed.
This usually means that you have TF>=2.14, or that you are loading a model that was trained on PCA but loaded with the log (or viceversa), or that you are loading a non-standard model from the cosmopower-organization repo.
Falling back to the dictionary, in case this also fails or does not output the right shape make sure you ran the `convert_tf214.py` script, and that a `.npz` file exists among the trained models, and that you ran `pip install .`. Also make sure that you are asking for the right probe between `custom_log` and `custom_pca`.
Tried to load pickle file from pre-trained model, but failed.
This usually means that you have TF>=2.14, or that you are loading a model that was trained on PCA but loaded with the log (or viceversa), or that you are loading a non-standard model from the cosmopower-organization repo.
Falling back to the dictionary, in case this also fails or does not output the right shape make sure you r

## Sampled basis and cosmology helpers

In [4]:
FIDUCIAL_100THETA = scalar_value(
    100.0 * bg.theta_star(
        FIDUCIAL_NATIVE['ombh2'],
        FIDUCIAL_NATIVE['omch2'],
        FIDUCIAL_NATIVE['H0'] / 100.0,
        mnu=MNU_FIXED,
        neff=NEFF_FIXED,
    )
)
FIDUCIAL_SAMPLED = {
    '100theta': FIDUCIAL_100THETA,
    'H0': FIDUCIAL_NATIVE['H0'],
    'ombh2': FIDUCIAL_NATIVE['ombh2'],
    'omch2': FIDUCIAL_NATIVE['omch2'],
    'logA': FIDUCIAL_NATIVE['logA'],
    'ns': FIDUCIAL_NATIVE['ns'],
    'tau': FIDUCIAL_NATIVE['tau'],
}


def solve_h_from_100theta(theta100, ombh2, omch2, *, mnu=MNU_FIXED, neff=NEFF_FIXED, n_iter=8):
    target = jnp.asarray(theta100, dtype=jnp.float64) / 100.0
    ombh2 = jnp.asarray(ombh2, dtype=jnp.float64)
    omch2 = jnp.asarray(omch2, dtype=jnp.float64)

    def body(h, _):
        def objective(hh):
            return bg.theta_star(ombh2, omch2, hh, mnu=mnu, neff=neff) - target

        f = objective(h)
        df = jax.grad(objective)(h)
        safe_df = jnp.where(jnp.abs(df) < 1.0e-8, jnp.sign(df) * 1.0e-8 + (df == 0.0) * 1.0e-8, df)
        delta = jnp.clip(f / safe_df, -0.08, 0.08)
        h_new = jnp.clip(h - delta, 0.4, 1.0)
        return h_new, None

    h0 = jnp.asarray(FIDUCIAL_NATIVE['H0'] / 100.0, dtype=jnp.float64)
    h_final, _ = jax.lax.scan(body, h0, xs=jnp.arange(n_iter))
    return h_final


def sampled_dict_from_array(theta, sampled_keys):
    theta = jnp.asarray(theta, dtype=jnp.float64)
    return {key: theta[i] for i, key in enumerate(sampled_keys)}


def native_cosmo_dict_from_sampled(theta_cosmo, sampled_keys):
    sampled = sampled_dict_from_array(theta_cosmo, sampled_keys)
    native = {key: jnp.asarray(FIDUCIAL_NATIVE[key], dtype=jnp.float64) for key in COSMO_KEYS_NATIVE}
    for key in ('ombh2', 'omch2', 'logA', 'ns', 'tau'):
        if key in sampled:
            native[key] = jnp.asarray(sampled[key], dtype=jnp.float64)
    if 'H0' in sampled:
        native['H0'] = jnp.asarray(sampled['H0'], dtype=jnp.float64)
    else:
        theta100 = sampled.get('100theta', jnp.asarray(FIDUCIAL_SAMPLED['100theta'], dtype=jnp.float64))
        native['H0'] = 100.0 * solve_h_from_100theta(theta100, native['ombh2'], native['omch2'])
    return native


def native_cosmo_array_from_sampled(theta_cosmo, sampled_keys):
    native = native_cosmo_dict_from_sampled(theta_cosmo, sampled_keys)
    return CosmoParams(native).to_array()


def omega_m_from_native(native_theta, *, mnu_fixed=MNU_FIXED):
    native_theta = jnp.asarray(native_theta, dtype=jnp.float64)
    h = native_theta[0] / 100.0
    return (native_theta[1] + native_theta[2] + mnu_fixed / 93.14) / h**2


def sigma8_from_native(native_theta):
    native_theta = jnp.asarray(native_theta, dtype=jnp.float64)
    H0, ombh2, omch2, logA, ns, tau = native_theta[:6]
    h = H0 / 100.0
    emulator_input = {
        'ombh2': jnp.atleast_1d(ombh2),
        'omch2': jnp.atleast_1d(omch2),
        'logA': jnp.atleast_1d(logA),
        'ns': jnp.atleast_1d(ns),
        'h': jnp.atleast_1d(h),
        'tau': jnp.atleast_1d(tau),
        'z': jnp.atleast_1d(jnp.asarray(PKLIN_FIXED_INPUTS['z'], dtype=jnp.float64)),
        'A_b': jnp.atleast_1d(jnp.asarray(PKLIN_FIXED_INPUTS['A_b'], dtype=jnp.float64)),
        'eta_b': jnp.atleast_1d(jnp.asarray(PKLIN_FIXED_INPUTS['eta_b'], dtype=jnp.float64)),
        'logT_AGN': jnp.atleast_1d(jnp.asarray(PKLIN_FIXED_INPUTS['logT_AGN'], dtype=jnp.float64)),
    }
    pklin = jnp.ravel(jnp.asarray(pklin_emulator.predict(emulator_input), dtype=jnp.float64))
    kmodes = jnp.asarray(pklin_emulator.modes, dtype=jnp.float64) / h
    return sigma8_from_linear_pk(kmodes, pklin)


def S8_from_native(native_theta):
    omega_m = omega_m_from_native(native_theta, mnu_fixed=MNU_FIXED)
    return sigma8_from_native(native_theta) * jnp.sqrt(omega_m / 0.3)


print(f'Fiducial 100theta_MC surrogate = {FIDUCIAL_100THETA:.8f}')
print(f'Solved back to H0 = {scalar_value(100.0 * solve_h_from_100theta(FIDUCIAL_100THETA, FIDUCIAL_NATIVE["ombh2"], FIDUCIAL_NATIVE["omch2"])):.6f}')
print(f'Fiducial sigma8 = {scalar_value(sigma8_from_native(CosmoParams(FIDUCIAL_NATIVE).to_array())):.6f}')
print(f'Fiducial Omega_m = {scalar_value(omega_m_from_native(CosmoParams(FIDUCIAL_NATIVE).to_array())):.6f}')


Fiducial 100theta_MC surrogate = 1.04016200
Solved back to H0 = 67.320000
Fiducial sigma8 = 0.810961
Fiducial Omega_m = 0.315567


## Likelihood terms and combination builder

In [5]:
def make_lcdm_bao_loglike_fn(bao_data, *, mnu_fixed=MNU_FIXED):
    bao_native_fn = make_bao_theory_fn(
        bao_data,
        cosmo_keys=('ombh2', 'omch2', 'h'),
        cosmo_sizes=(1, 1, 1),
        mnu_fixed=mnu_fixed,
    )
    data = jnp.asarray(bao_data.data_vector, dtype=jnp.float64)
    cov_inv = jnp.asarray(inv(bao_data.cov), dtype=jnp.float64)

    def bao_loglike_native(native_theta):
        native_theta = jnp.asarray(native_theta, dtype=jnp.float64)
        H0, ombh2, omch2 = native_theta[0], native_theta[1], native_theta[2]
        bao_params = jnp.array([ombh2, omch2, H0 / 100.0], dtype=jnp.float64)
        theory = bao_native_fn(bao_params)
        residual = data - theory
        return -0.5 * residual @ cov_inv @ residual

    return jax.jit(bao_loglike_native)


bao_loglike_native = make_lcdm_bao_loglike_fn(bao_data, mnu_fixed=MNU_FIXED)


def nuisance_scale(default):
    default = float(default)
    return max(abs(default) * 0.1, 1.0e-3)


def combo_prior_logp(prior_mode, theta_cosmo, sampled_keys, native_theta):
    sampled = sampled_dict_from_array(theta_cosmo, sampled_keys)
    H0 = native_theta[0]
    ombh2 = native_theta[1]
    omch2 = native_theta[2]
    logA = native_theta[3]
    ns = native_theta[4]
    tau = native_theta[5]

    inside_common = (
        (40.0 <= H0) & (H0 <= 100.0)
        & (0.005 <= omch2) & (omch2 <= 0.99)
        & (1.61 <= logA) & (logA <= 4.0)
    )
    if '100theta' in sampled_keys:
        theta100 = sampled['100theta']
        inside_common = inside_common & ((0.5 <= theta100) & (theta100 <= 10.0))

    if prior_mode == 'planck_aniso':
        inside = (
            inside_common
            & (0.005 <= ombh2) & (ombh2 <= 0.1)
            & (0.8 <= ns) & (ns <= 1.2)
            & (0.01 <= tau) & (tau <= 0.8)
        )
        return jnp.where(inside, 0.0, -jnp.inf)

    if prior_mode in {'lensing_bbn', 'bao_bbn'}:
        inside = inside_common & (0.85 <= ns) & (ns <= 1.10)
        gauss = -0.5 * ((ombh2 - MOSSABBN_MEAN) / MOSSABBN_SIGMA) ** 2
        gauss += -0.5 * ((ns - NS_PRIOR_MEAN) / NS_PRIOR_SIGMA) ** 2
        return jnp.where(inside, gauss, -jnp.inf)

    raise ValueError(f'unknown prior_mode {prior_mode!r}')


NONPLANCK_THETA_FID = jnp.asarray(
    [FIDUCIAL_SAMPLED[key] for key in NONPLANCK_SAMPLED_COSMO_KEYS],
    dtype=jnp.float64,
)
PLANCK_THETA_FID = jnp.asarray(
    [FIDUCIAL_SAMPLED[key] for key in PLANCK_SAMPLED_COSMO_KEYS],
    dtype=jnp.float64,
)
NONPLANCK_FALLBACK_SCALES = jnp.asarray(
    [COSMO_FALLBACK_SCALES[key] for key in NONPLANCK_SAMPLED_COSMO_KEYS],
    dtype=jnp.float64,
)
PLANCK_FALLBACK_SCALES = jnp.asarray(
    [COSMO_FALLBACK_SCALES[key] for key in PLANCK_SAMPLED_COSMO_KEYS],
    dtype=jnp.float64,
)
MASTER_NUISANCE_DEFAULTS = jnp.asarray(
    [nuisance_defaults[name] for name in MASTER_CMB_NUISANCE_NAMES],
    dtype=jnp.float64,
) if MASTER_CMB_NUISANCE_NAMES else jnp.zeros((0,), dtype=jnp.float64)
MASTER_NUISANCE_FALLBACK_SCALES = jnp.asarray(
    [nuisance_scale(nuisance_defaults[name]) for name in MASTER_CMB_NUISANCE_NAMES],
    dtype=jnp.float64,
) if MASTER_CMB_NUISANCE_NAMES else jnp.zeros((0,), dtype=jnp.float64)


def term_weights_for(term_names):
    weights = np.zeros(len(MASTER_CMB_TERM_ORDER), dtype=np.float64)
    for term_name in term_names:
        weights[TERM_INDEX[term_name]] = 1.0
    return jnp.asarray(weights, dtype=jnp.float64)


def sample_to_native_and_layout(theta_varied, sampled_keys):
    theta_varied = jnp.asarray(theta_varied, dtype=jnp.float64)
    n_cosmo = len(sampled_keys)
    theta_cosmo = theta_varied[:n_cosmo]
    theta_nuis = theta_varied[n_cosmo:]
    native_dict = native_cosmo_dict_from_sampled(theta_cosmo, sampled_keys)
    native_cosmo = CosmoParams(native_dict)
    nuisance_map = {
        name: theta_nuis[i]
        for i, name in enumerate(MASTER_CMB_NUISANCE_NAMES)
    }
    layout_theta = MASTER_LAYOUT.pack(native_cosmo, nuisance_map)
    return theta_cosmo, native_cosmo.to_array(), layout_theta


def build_nonplanck_log_post(*, include_bao, prior_mode, cmb_term_names):
    term_weights = term_weights_for(cmb_term_names)
    include_bao_flag = jnp.asarray(include_bao)

    @jax.jit
    def log_post(theta_varied):
        theta_cosmo, native_theta, layout_theta = sample_to_native_and_layout(
            theta_varied,
            NONPLANCK_SAMPLED_COSMO_KEYS,
        )
        logp = combo_prior_logp(
            prior_mode,
            theta_cosmo,
            NONPLANCK_SAMPLED_COSMO_KEYS,
            native_theta,
        )
        logp = logp + jax.lax.cond(
            include_bao_flag,
            lambda _: bao_loglike_native(native_theta),
            lambda _: jnp.asarray(0.0, dtype=jnp.float64),
            operand=None,
        )
        for idx, term_name in enumerate(MASTER_CMB_TERM_ORDER):
            logp = logp + jax.lax.cond(
                term_weights[idx] > 0.0,
                lambda _: CMB_TERM_LOGLIKES[term_name](layout_theta),
                lambda _: jnp.asarray(0.0, dtype=jnp.float64),
                operand=None,
            )
        return logp

    return log_post


PLANCK_PRIMARY_LOG_POST = jax.jit(
    lambda theta_varied: (
        lambda theta_cosmo, native_theta, layout_theta: combo_prior_logp(
            'planck_aniso',
            theta_cosmo,
            PLANCK_SAMPLED_COSMO_KEYS,
            native_theta,
        )
        + CMB_TERM_LOGLIKES['planck_highl'](layout_theta)
        + CMB_TERM_LOGLIKES['planck_lowl_ee'](layout_theta)
    )(*sample_to_native_and_layout(theta_varied, PLANCK_SAMPLED_COSMO_KEYS))
)


NONPLANCK_LOG_POSTS = {
    'BAO + BBN': build_nonplanck_log_post(
        include_bao=True,
        prior_mode='bao_bbn',
        cmb_term_names=(),
    ),
    'ACT DR6 lensing + BBN': build_nonplanck_log_post(
        include_bao=False,
        prior_mode='lensing_bbn',
        cmb_term_names=('act_dr6_lensing',),
    ),
    'Planck lensing + BAO + BBN': build_nonplanck_log_post(
        include_bao=True,
        prior_mode='lensing_bbn',
        cmb_term_names=('planck_lensing',),
    ),
    'ACT DR6 lensing + BAO + BBN': build_nonplanck_log_post(
        include_bao=True,
        prior_mode='lensing_bbn',
        cmb_term_names=('act_dr6_lensing',),
    ),
    'ACT + Planck lensing + BAO + BBN': build_nonplanck_log_post(
        include_bao=True,
        prior_mode='lensing_bbn',
        cmb_term_names=('act_dr6_lensing', 'planck_lensing'),
    ),
}


def build_combination(label, *, sampled_cosmo_keys, cmb_term_names=(), include_bao=False, prior_mode='lensing_bbn'):
    cmb_term_names = tuple(cmb_term_names)
    if sampled_cosmo_keys == PLANCK_SAMPLED_COSMO_KEYS:
        theta_fid = PLANCK_THETA_FID
        fallback_scales = PLANCK_FALLBACK_SCALES
        log_post = PLANCK_PRIMARY_LOG_POST
        sampled_to_native_and_layout = lambda theta_varied: sample_to_native_and_layout(theta_varied, PLANCK_SAMPLED_COSMO_KEYS)
    elif sampled_cosmo_keys == NONPLANCK_SAMPLED_COSMO_KEYS:
        theta_fid = NONPLANCK_THETA_FID
        fallback_scales = NONPLANCK_FALLBACK_SCALES
        log_post = NONPLANCK_LOG_POSTS[label]
        sampled_to_native_and_layout = lambda theta_varied: sample_to_native_and_layout(theta_varied, NONPLANCK_SAMPLED_COSMO_KEYS)
    else:
        raise ValueError(f'unexpected sampled_cosmo_keys {sampled_cosmo_keys!r}')

    theta_fid = jnp.concatenate([theta_fid, MASTER_NUISANCE_DEFAULTS]) if MASTER_NUISANCE_DEFAULTS.size else theta_fid
    fallback_scales = jnp.concatenate([fallback_scales, MASTER_NUISANCE_FALLBACK_SCALES]) if MASTER_NUISANCE_FALLBACK_SCALES.size else fallback_scales

    fid_theta_cosmo, fid_native, fid_layout = sampled_to_native_and_layout(theta_fid)
    fid_terms = {}
    if include_bao:
        fid_terms['bao'] = scalar_value(bao_loglike_native(fid_native))
    for term_name in cmb_term_names:
        fid_terms[term_name] = scalar_value(CMB_TERM_LOGLIKES[term_name](fid_layout))

    return {
        'label': label,
        'sampled_cosmo_keys': tuple(sampled_cosmo_keys),
        'cmb_term_names': cmb_term_names,
        'cmb_nuisance_names': tuple(MASTER_CMB_NUISANCE_NAMES),
        'theta_fid': theta_fid,
        'fallback_scales': fallback_scales,
        'log_post': log_post,
        'sampled_to_native_and_layout': sampled_to_native_and_layout,
        'fid_terms': fid_terms,
        'fid_native': fid_native,
        'prior_mode': prior_mode,
        'include_bao': include_bao,
    }


combination_specs = {}
if RUN_BAO_BBN:
    combination_specs['BAO + BBN'] = build_combination(
        'BAO + BBN',
        sampled_cosmo_keys=NONPLANCK_SAMPLED_COSMO_KEYS,
        cmb_term_names=(),
        include_bao=True,
        prior_mode='bao_bbn',
    )
if RUN_PLANCK_PRIMARY:
    combination_specs['Planck anisotropies'] = build_combination(
        'Planck anisotropies',
        sampled_cosmo_keys=PLANCK_SAMPLED_COSMO_KEYS,
        cmb_term_names=('planck_highl', 'planck_lowl_ee'),
        include_bao=False,
        prior_mode='planck_aniso',
    )
if RUN_ACT_LENS_BBN:
    combination_specs['ACT DR6 lensing + BBN'] = build_combination(
        'ACT DR6 lensing + BBN',
        sampled_cosmo_keys=NONPLANCK_SAMPLED_COSMO_KEYS,
        cmb_term_names=('act_dr6_lensing',),
        include_bao=False,
        prior_mode='lensing_bbn',
    )
if RUN_PLANCK_LENSING_BAO:
    combination_specs['Planck lensing + BAO + BBN'] = build_combination(
        'Planck lensing + BAO + BBN',
        sampled_cosmo_keys=NONPLANCK_SAMPLED_COSMO_KEYS,
        cmb_term_names=('planck_lensing',),
        include_bao=True,
        prior_mode='lensing_bbn',
    )
if RUN_ACT_LENSING_BAO:
    combination_specs['ACT DR6 lensing + BAO + BBN'] = build_combination(
        'ACT DR6 lensing + BAO + BBN',
        sampled_cosmo_keys=NONPLANCK_SAMPLED_COSMO_KEYS,
        cmb_term_names=('act_dr6_lensing',),
        include_bao=True,
        prior_mode='lensing_bbn',
    )
if RUN_ACT_PLANCK_LENSING_BAO:
    combination_specs['ACT + Planck lensing + BAO + BBN'] = build_combination(
        'ACT + Planck lensing + BAO + BBN',
        sampled_cosmo_keys=NONPLANCK_SAMPLED_COSMO_KEYS,
        cmb_term_names=('act_dr6_lensing', 'planck_lensing'),
        include_bao=True,
        prior_mode='lensing_bbn',
    )

summary_rows = []
for label, spec in combination_specs.items():
    summary_rows.append({
        'dataset': label,
        'sampled_cosmo_keys': ', '.join(spec['sampled_cosmo_keys']),
        'cmb_nuisance_count': len(spec['cmb_nuisance_names']),
        'cmb_terms': ', '.join(spec['cmb_term_names']) if spec['cmb_term_names'] else 'none',
        'includes_bao': spec['include_bao'],
        'prior_mode': spec['prior_mode'],
        'log_post(theta_fid)': scalar_value(spec['log_post'](spec['theta_fid'])),
    })

display(pd.DataFrame(summary_rows))

for label, spec in combination_specs.items():
    print(label)
    print('  sampled cosmology:', spec['sampled_cosmo_keys'])
    print('  sampled CMB nuisance count:', len(spec['cmb_nuisance_names']))
    print('  fiducial term contributions:')
    for term_name, value in spec['fid_terms'].items():
        print(f'    {term_name:>20s}: {value:.6f}')
    print(f'  prior contribution: {scalar_value(combo_prior_logp(spec["prior_mode"], spec["theta_fid"][:len(spec["sampled_cosmo_keys"])], spec["sampled_cosmo_keys"], spec["fid_native"])):.6f}')
    print()



,dataset,sampled_cosmo_keys,cmb_nuisance_count,cmb_terms,includes_bao,prior_mode,log_post(theta_fid)
0,BAO + BBN,"100theta, ombh2, omch2, logA, ns",0,none,True,bao_bbn,-16.036089
1,Planck anisotropies,"100theta, ombh2, omch2, logA, ns, tau",0,"planck_highl, planck_lowl_ee",False,planck_aniso,-1372.625831
2,ACT DR6 lensing + BBN,"100theta, ombh2, omch2, logA, ns",0,act_dr6_lensing,False,lensing_bbn,-7.085057
3,Planck lensing + BAO + BBN,"100theta, ombh2, omch2, logA, ns",0,planck_lensing,True,lensing_bbn,-20.545822
4,ACT DR6 lensing + BAO + BBN,"100theta, ombh2, omch2, logA, ns",0,act_dr6_lensing,True,lensing_bbn,-23.084961
5,ACT + Planck lensing + BAO + BBN,"100theta, ombh2, omch2, logA, ns",0,"act_dr6_lensing, planck_lensing",True,lensing_bbn,-27.594694


BAO + BBN
  sampled cosmology: ('100theta', 'ombh2', 'omch2', 'logA', 'ns')
  sampled CMB nuisance count: 0
  fiducial term contributions:
                     bao: -15.999904
  prior contribution: -0.036185

Planck anisotropies
  sampled cosmology: ('100theta', 'ombh2', 'omch2', 'logA', 'ns', 'tau')
  sampled CMB nuisance count: 0
  fiducial term contributions:
            planck_highl: -1174.547951
          planck_lowl_ee: -198.077879
  prior contribution: 0.000000

ACT DR6 lensing + BBN
  sampled cosmology: ('100theta', 'ombh2', 'omch2', 'logA', 'ns')
  sampled CMB nuisance count: 0
  fiducial term contributions:
         act_dr6_lensing: -7.048872
  prior contribution: -0.036185

Planck lensing + BAO + BBN
  sampled cosmology: ('100theta', 'ombh2', 'omch2', 'logA', 'ns')
  sampled CMB nuisance count: 0
  fiducial term contributions:
                     bao: -15.999904
          planck_lensing: -4.509733
  prior contribution: -0.036185

ACT DR6 lensing + BAO + BBN
  sampled cosmol

## Run NUTS for each combination

In [ ]:
def whitening_scales_from_hessian(log_post_physical, theta_fid, fallback_scales):
    nll = lambda theta: -log_post_physical(theta)
    hess = np.asarray(jax.jit(jax.hessian(nll))(theta_fid), dtype=float)
    fisher = 0.5 * (hess + hess.T)
    cov = np.linalg.pinv(fisher)
    scales = np.sqrt(np.clip(np.diag(cov), 0.0, np.inf))
    fallback = np.asarray(fallback_scales, dtype=float)
    bad = (~np.isfinite(scales)) | (scales <= 0.0)
    scales[bad] = fallback[bad]
    scales = np.maximum(scales, 0.20 * fallback)
    return fisher, cov, scales


def manual_scales_for_spec(spec):
    entries = MANUAL_WHITENING_SCALES.get(spec['label'])
    if not entries:
        return None
    return jnp.asarray([entries[key] for key in spec['sampled_cosmo_keys']], dtype=jnp.float64)


def run_combination_chain(spec, *, seed):
    manual_scales = manual_scales_for_spec(spec)
    if manual_scales is None:
        fisher_local, cov_local, scales = whitening_scales_from_hessian(
            spec['log_post'],
            spec['theta_fid'],
            spec['fallback_scales'],
        )
    else:
        fisher_local = None
        cov_local = None
        scales = manual_scales
    to_whitened, to_physical = make_transform(center=spec['theta_fid'], scale=scales)

    @jax.jit
    def log_post_whitened(x):
        return spec['log_post'](to_physical(x))

    x0 = jnp.zeros_like(spec['theta_fid'])
    samples_w, diagnostics = run_nuts(
        jax.random.key(seed),
        log_post_whitened,
        initial_position=x0,
        num_warmup=NUM_WARMUP,
        num_samples=NUM_SAMPLES,
        num_chains=NUM_CHAINS,
        adapt_mass_matrix=False,
        mass_matrix_type='diagonal',
        initial_inverse_mass_matrix=jnp.ones(spec['theta_fid'].shape[0]),
        max_tree_depth=MAX_TREE_DEPTH,
        scan_chunk_size=SCAN_CHUNK,
        parallel_chains=True,
    )
    samples_phys = samples_to_physical(samples_w, to_physical)
    flat = np.asarray(samples_phys).reshape(-1, spec['theta_fid'].shape[0])
    flat_log_post = np.asarray(jax.vmap(spec['log_post'])(samples_phys.reshape(-1, spec['theta_fid'].shape[0])))
    flat_divergent = np.asarray(diagnostics['is_divergent']).reshape(-1).astype(bool)
    valid = ~flat_divergent
    flat_valid = flat[valid] if np.any(valid) else flat
    flat_log_post_valid = flat_log_post[valid] if np.any(valid) else flat_log_post
    map_idx = int(np.argmax(flat_log_post_valid))
    return {
        'spec': spec,
        'samples_w': np.asarray(samples_w),
        'samples_phys': np.asarray(samples_phys),
        'flat_samples': flat_valid,
        'flat_log_post': flat_log_post_valid,
        'diagnostics': diagnostics,
        'map_theta': flat_valid[map_idx],
        'whitening_scales': scales,
        'fisher_local': fisher_local,
        'cov_local': cov_local,
    }


results = {}
seed_base = 40
for offset, (label, spec) in enumerate(combination_specs.items()):
    print(f'Running {label} ...')
    results[label] = run_combination_chain(spec, seed=seed_base + offset)
    print(f'  sampled dimension = {results[label]["flat_samples"].shape[1]}')
    print(f'  whitening scales  = {results[label]["whitening_scales"]}')
    print()



Running BAO + BBN ...
  sampled dimension = 5
  whitening scales  = [0.005      0.00032479 0.006      0.05       0.02      ]

Running Planck anisotropies ...
  sampled dimension = 6
  whitening scales  = [0.001      0.00014263 0.00147736 0.05400449 0.004      0.02779918]

Running ACT DR6 lensing + BBN ...
  sampled dimension = 5
  whitening scales  = [0.02    0.00045 0.02    0.12    0.03   ]

Running Planck lensing + BAO + BBN ...


## Diagnostics and derived parameters

In [ ]:
diagnostic_rows = []
for label, result in results.items():
    accept = np.asarray(result['diagnostics']['acceptance_rate'])
    n_steps = np.asarray(result['diagnostics']['num_integration_steps'])
    divergent = np.asarray(result['diagnostics']['is_divergent'])
    diagnostic_rows.append({
        'dataset': label,
        'accept_mean': float(accept.mean()),
        'steps_mean': float(n_steps.mean()),
        'divergent_total': int(divergent.sum()),
    })
    print(label)
    for chain in range(accept.shape[0]):
        print(
            f'  chain {chain + 1}: '
            f'accept={float(accept[chain].mean()):.3f}  '
            f'steps={float(n_steps[chain].mean()):.1f}  '
            f'divergent={int(divergent[chain].sum())}'
        )
    print()

display(pd.DataFrame(diagnostic_rows))


@jax.jit
def native_batch_nonplanck(samples):
    return jax.vmap(lambda row: native_cosmo_array_from_sampled(row, NONPLANCK_SAMPLED_COSMO_KEYS))(samples)


@jax.jit
def native_batch_planck(samples):
    return jax.vmap(lambda row: native_cosmo_array_from_sampled(row, PLANCK_SAMPLED_COSMO_KEYS))(samples)


@jax.jit
def omega_m_batch(native_samples):
    return jax.vmap(omega_m_from_native)(native_samples)


@jax.jit
def sigma8_batch(native_samples):
    return jax.vmap(sigma8_from_native)(native_samples)


@jax.jit
def s8_batch(native_samples):
    return jax.vmap(S8_from_native)(native_samples)


def batched_eval_in_chunks(values, fn, chunk_size=DERIVED_CHUNK_SIZE):
    values = np.asarray(values)
    outputs = []
    for start in range(0, len(values), chunk_size):
        stop = min(start + chunk_size, len(values))
        chunk = jnp.asarray(values[start:stop], dtype=jnp.float64)
        outputs.append(np.asarray(fn(chunk)))
    if not outputs:
        return np.empty((0,), dtype=float)
    return np.concatenate(outputs, axis=0)


def derived_dataframe(result):
    spec = result['spec']
    n_cosmo = len(spec['sampled_cosmo_keys'])
    sampled_cosmo = np.asarray(result['flat_samples'][:, :n_cosmo], dtype=float)
    native_batch_fn = native_batch_planck if spec['sampled_cosmo_keys'] == PLANCK_SAMPLED_COSMO_KEYS else native_batch_nonplanck
    native = batched_eval_in_chunks(sampled_cosmo, native_batch_fn)
    omega_m = batched_eval_in_chunks(native, omega_m_batch)
    sigma8 = batched_eval_in_chunks(native, sigma8_batch)
    s8 = batched_eval_in_chunks(native, s8_batch)
    h0 = np.asarray(native[:, 0])
    return pd.DataFrame({
        'Omega_m': omega_m,
        'sigma8': sigma8,
        'S8': s8,
        'H0': h0,
    })


derived_results = {label: derived_dataframe(result) for label, result in results.items()}
summary_rows = []
for label, df in derived_results.items():
    row = {'dataset': label}
    for key in ('Omega_m', 'sigma8', 'S8', 'H0'):
        stats = summarize_scalar(df[key])
        row[f'{key}_mean'] = stats['mean']
        row[f'{key}_std'] = stats['std']
    summary_rows.append(row)

display(pd.DataFrame(summary_rows))



## `sigma8`-`Omega_m` comparison plot

In [ ]:
def density_levels_from_hist(hist, credible_levels=(0.68, 0.95)):
    flat = hist.ravel()
    flat = flat[np.isfinite(flat)]
    if flat.size == 0 or np.all(flat <= 0.0):
        return []
    order = np.argsort(flat)[::-1]
    sorted_hist = flat[order]
    cdf = np.cumsum(sorted_hist)
    cdf /= cdf[-1]
    levels = []
    for cl in credible_levels:
        idx = np.searchsorted(cdf, cl, side='left')
        idx = min(idx, sorted_hist.size - 1)
        levels.append(sorted_hist[idx])
    return sorted(set(levels))


def plot_smoothed_contours(
    ax,
    x_samples,
    y_samples,
    color,
    *,
    bins=80,
    label=None,
    x_range=None,
    y_range=None,
    smoothing_sigma=1.1,
    fill_alphas=(0.10, 0.24),
    lw=(1.0, 1.6),
    zorder=3,
):
    hist, xedges, yedges = np.histogram2d(
        np.asarray(x_samples),
        np.asarray(y_samples),
        bins=bins,
        range=[x_range, y_range],
    )
    hist = gaussian_filter(hist, sigma=smoothing_sigma)
    levels = density_levels_from_hist(hist)
    if levels:
        xcenters = 0.5 * (xedges[:-1] + xedges[1:])
        ycenters = 0.5 * (yedges[:-1] + yedges[1:])
        ax.contourf(
            xcenters,
            ycenters,
            hist.T,
            levels=[levels[0], hist.max()],
            colors=[color],
            alpha=fill_alphas[0],
            zorder=zorder,
        )
        ax.contourf(
            xcenters,
            ycenters,
            hist.T,
            levels=[levels[-1], hist.max()],
            colors=[color],
            alpha=fill_alphas[1],
            zorder=zorder + 0.1,
        )
        ax.contour(
            xcenters,
            ycenters,
            hist.T,
            levels=levels,
            colors=[color],
            linewidths=lw[:len(levels)],
            zorder=zorder + 0.2,
        )
    if label is not None:
        ax.plot([], [], color=color, lw=lw[-1], label=label)


plot_order = [
    'BAO + BBN',
    'Planck anisotropies',
    'ACT DR6 lensing + BBN',
    'Planck lensing + BAO + BBN',
    'ACT DR6 lensing + BAO + BBN',
    'ACT + Planck lensing + BAO + BBN',
]
plot_order = [label for label in plot_order if label in derived_results]

xmin = min(float(derived_results[label]['Omega_m'].quantile(0.01)) for label in plot_order)
xmax = max(float(derived_results[label]['Omega_m'].quantile(0.99)) for label in plot_order)
ymin = min(float(derived_results[label]['sigma8'].quantile(0.01)) for label in plot_order)
ymax = max(float(derived_results[label]['sigma8'].quantile(0.99)) for label in plot_order)
xpad = 0.06 * (xmax - xmin)
ypad = 0.06 * (ymax - ymin)
x_range = (xmin - xpad, xmax + xpad)
y_range = (ymin - ypad, ymax + ypad)

fig, ax = plt.subplots(figsize=(7.2, 5.8))
for label in plot_order:
    df = derived_results[label]
    plot_smoothed_contours(
        ax,
        df['Omega_m'],
        df['sigma8'],
        COMBINATION_COLORS[label],
        label=label,
        x_range=x_range,
        y_range=y_range,
    )

if INCLUDE_PAPER_REFERENCE_MARKERS:
    for label, values in PAPER_REFERENCE_POINTS.items():
        marker_color = values.get('color', COMBINATION_COLORS.get(label, 'black'))
        ax.scatter(
            values['Omega_m'],
            values['sigma8'],
            marker='x',
            color=marker_color,
            s=50,
            linewidths=1.5,
            zorder=6,
        )
        if 'extended' in label:
            ax.text(
                values['Omega_m'] + 0.0008,
                values['sigma8'] + 0.0008,
                'ext.',
                color=marker_color,
                fontsize=9,
                zorder=6,
            )

ax.set_xlabel(PARAM_LABELS['Omega_m'])
ax.set_ylabel(PARAM_LABELS['sigma8'])
ax.set_title(r'LCDM comparison in the $\sigma_8$-$\Omega_m$ plane')
ax.set_xlim(*x_range)
ax.set_ylim(*y_range)
ax.grid(alpha=0.25)
ax.legend(frameon=False, loc='best')
plt.tight_layout()
plt.show()


## Notes

- `Planck anisotropies` uses the paper-matched primary-CMB stack: Planck high-ell `TTTEEE` plus low-ell `EE`, with no BBN prior added on top.
- The non-Planck-primary combinations use the Mossa et al. (2020) BBN prior `ombh2 = 0.02233 +/- 0.00036` and the paper's Gaussian `ns` prior `0.96 +/- 0.02`.
- `BAO + BBN` is included because it helps check the distance pipeline and the `100theta` parameterization, but its `sigma8` projection is necessarily prior-dominated in this notebook because pure BAO does not constrain the primordial amplitude.
- If you want to compare directly to the paper's quoted one-dimensional values, the most meaningful rows are `Planck anisotropies` and `ACT DR6 lensing + BAO + BBN`, which have Table 2 reference points above.
